# Wykład 04 — przykłady

## Obsługa błędów: else i finally - slajd 4

Dla liczbowych danych sprawdzamy, czy czas pomiaru jest dodatni. Niepoprawna wartość powoduje `ValueError`, który przechwytujemy w miejscu wywołania.

* Który blok wykona się dla dodatniego czasu, a który dla zera lub liczby ujemnej? Czy `finally` wykona się w każdej z tych prób?


In [ ]:
def czestosc_zliczen(liczba_zliczen, czas_s):
    if czas_s <= 0:
        raise ValueError("Czas pomiaru musi być dodatni.")
    return liczba_zliczen / czas_s

for czas in (2.0, 0.0, -1.0):
    print(f"Czas pomiaru: {czas} s")
    try:
        wynik = czestosc_zliczen(120, czas)
    except ValueError as blad:
        print("Błąd danych:", blad)
    else:
        print(f"Wynik: {wynik:.1f} zliczeń/s")
    finally:
        print("Koniec próby.\n")

## Rekurencja a iteracja: silnia - slajd 13

Zakładamy, że `n` jest liczbą całkowitą nieujemną. Korzystamy z zależności `n! = n · (n - 1)!` oraz warunków kończących `0! = 1` i `1! = 1`. Obie funkcje rozwiązują ten sam problem.

* Co zatrzymuje rekurencję? Jaką rolę w rozwiązaniu iteracyjnym pełni zmienna `wynik`?


In [ ]:
def silnia_rekurencja(n):
    if n <= 1:
        return 1
    return n * silnia_rekurencja(n - 1)


def silnia_iteracja(n):
    wynik = 1
    for i in range(2, n + 1):
        wynik *= i
    return wynik


for n in (0, 1, 5, 10):
    rekurencyjnie = silnia_rekurencja(n)
    iteracyjnie = silnia_iteracja(n)
    print(f"{n}! = {rekurencyjnie}; iteracyjnie: {iteracyjnie}")
    assert rekurencyjnie == iteracyjnie

## Limit rekurencji i stos wywołań - slajd 17

Każde kolejne wywołanie `silnia_rekurencja()` tworzy ramkę na stosie wywołań. Poprzednie wywołanie czeka na wynik, pamiętając swoje `n` i miejsce dalszego wykonania. Dla `silnia_rekurencja(3)` wywołania schodzą przez `3 → 2 → 1`, a następnie wyniki wracają do oczekujących wywołań.

Limit głębokości rekurencji chroni stos interpretera. Przekroczenie limitu powoduje `RecursionError` - nie oznacza wyczerpania całej pamięci RAM. Część limitu zajmują już wywołania środowiska Jupyter, dlatego błąd może pojawić się dla `n` mniejszego niż sam limit. 

* Dlaczego pętla oblicza tę samą silnię bez tworzenia kolejnych zagnieżdżonych wywołań? Czy wersja iteracyjna również potrzebuje pamięci na coraz większy wynik?

In [ ]:
import math
import sys

def silnia_rekurencja(n):
    if n <= 1:
        return 1
    return n * silnia_rekurencja(n - 1)

def silnia_iteracja(n):
    wynik = 1
    for i in range(2, n + 1):
        wynik *= i
    return wynik

# Tak jak wcześniej: n jest liczbą całkowitą nieujemną.
limit = sys.getrecursionlimit()
n = limit + 50
print("Limit głębokości rekurencji:", limit)
print("Argument n:", n)

try:
    wynik = silnia_rekurencja(n)
    print("Wersja rekurencyjna zakończyła obliczenia.")
    print("Zgodność z math.factorial:", wynik == math.factorial(n))
    print("Liczba bitów wyniku:", wynik.bit_length())
except RecursionError:
    print("RecursionError: przekroczono limit głębokości wywołań.")

wynik = silnia_iteracja(n)
print("Wersja iteracyjna zakończyła obliczenia.")
print("Zgodność z math.factorial:", wynik == math.factorial(n))
# Nie wypisujemy ogromnej liczby; podajemy długość jej zapisu binarnego.
print("Liczba bitów wyniku:", wynik.bit_length())

## Sortowanie z wyrażeniem lambda - slajd 20

* Dlaczego `lambda probka: probka[1]` ustawia kolejność według odczytów? Jak posortować te same dane według nazw?

In [ ]:
probki = [("Próbka B", 1.30), ("Próbka A", 1.10), ("Próbka C", 1.20)]

wedlug_odczytu = sorted(probki, key=lambda probka: probka[1])
wedlug_nazwy = sorted(probki, key=lambda probka: probka[0])

print("Dane początkowe:", probki)
print("Według odczytu:", wedlug_odczytu)
print("Według nazwy:", wedlug_nazwy)

## map, filter i wyrażenia listowe - slajd 25

* Dlaczego druga próba odczytania tego samego iteratora daje pustą listę?

In [ ]:
odczyty_mV = [0.8, 1.2, 1.5, 0.9]

przeliczone = map(lambda odczyt: odczyt / 1000, odczyty_mV)
wynik_map = list(przeliczone)
print("map — odczyty w V:", wynik_map)
print("Ten sam iterator ponownie:", list(przeliczone))

wybrane = filter(lambda odczyt: odczyt >= 1.0, odczyty_mV)
wynik_filter = list(wybrane)
print("filter — odczyty >= 1.0 mV:", wynik_filter)
print("Ten sam iterator ponownie:", list(wybrane))

wynik_listowy_V = [odczyt / 1000 for odczyt in odczyty_mV]
wynik_listowy_wybrane = [odczyt for odczyt in odczyty_mV if odczyt >= 1.0]
print("Wyrażenie listowe — odczyty w V:", wynik_listowy_V)
print("Wyrażenie listowe — wybrane odczyty:", wynik_listowy_wybrane)
assert wynik_map == wynik_listowy_V
assert wynik_filter == wynik_listowy_wybrane

## Zmienna globalna i UnboundLocalError - slajd 37

* Dlaczego pierwsza funkcja zgłasza `UnboundLocalError`, mimo że poza nią istnieje `licznik`? Co zmienia deklaracja `global`?

In [ ]:
licznik = 0

def zwieksz_blednie():
    licznik += 1
    return licznik

try:
    zwieksz_blednie()
except UnboundLocalError:
    print("UnboundLocalError: próba odczytu lokalnego licznika przed przypisaniem.")

print("Licznik po nieudanej próbie:", licznik)

def zwieksz_globalny():
    global licznik
    licznik += 1
    return licznik

print("Wynik poprawionej funkcji:", zwieksz_globalny())
print("Licznik poza funkcją:", licznik)

## Czytanie pliku linia po linii - slajd 39

Przy przetwarzaniu kolejnych wierszy nie przechowujemy całej zawartości pliku. Nadal potrzebujemy pamięci na bieżący wiersz (w tym najdłuższy wiersz pliku), bufor odczytu i wyniki obliczeń. Przykład działa w katalogu tymczasowym, który zostanie automatycznie usunięty.

* Który sposób wybrać dla pliku z milionem wierszy, gdy potrzebujemy tylko liczby odczytów i ich sumy?

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as katalog:
    sciezka = Path(katalog) / "odczyty.txt"
    with open(sciezka, "w", encoding="utf-8") as plik:
        plik.write("1.25\n1.50\n1.75\n")

    with open(sciezka, "r", encoding="utf-8") as plik:
        wiersze = plik.readlines()
    liczba_z_listy = len(wiersze)
    suma_z_listy = sum(float(wiersz) for wiersz in wiersze)
    print("readlines — liczba:", liczba_z_listy, "suma:", suma_z_listy)
    del wiersze  # Lista wszystkich wierszy nie jest już potrzebna.

    liczba = 0
    suma = 0.0
    with open(sciezka, "r", encoding="utf-8") as plik:
        for wiersz in plik:
            suma += float(wiersz)
            liczba += 1
    print("Linia po linii — liczba:", liczba, "suma:", suma)
    assert (liczba, suma) == (liczba_z_listy, suma_z_listy)

## Ochrona pliku przed nadpisaniem - slajd 40

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as katalog:
    sciezka = Path(katalog) / "wyniki.txt"
    with open(sciezka, "w", encoding="utf-8") as plik:
        plik.write("Pierwszy pomiar: 1.2 mV\n")
    print("Początkowa treść:", sciezka.read_text(encoding="utf-8").strip())

    with open(sciezka, "w", encoding="utf-8") as plik:
        plik.write("Nowy pomiar: 1.4 mV\n")
    print("Po otwarciu w trybie w:", sciezka.read_text(encoding="utf-8").strip())

    zachowana_tresc = sciezka.read_text(encoding="utf-8")
    try:
        with open(sciezka, "x", encoding="utf-8") as plik:
            plik.write("Ta treść nie zostanie zapisana.\n")
    except FileExistsError:
        print("FileExistsError: plik już istnieje, zachowano jego zawartość.")
    assert sciezka.read_text(encoding="utf-8") == zachowana_tresc

    with open(sciezka, "a", encoding="utf-8") as plik:
        plik.write("Kolejny pomiar: 1.6 mV\n")
    print("Po dopisaniu w trybie a:")
    print(sciezka.read_text(encoding="utf-8"), end="")